# GridSight focused energy-market EDA

This notebook presents the tested Step 5.2 analysis over PostgreSQL reporting views. Reusable querying, validation, ranking, and figure logic remains under `src/gridsight/reporting/`. The analysis is descriptive: correlations are associations, not causal estimates.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from gridsight.database.connection import create_database_engine
from gridsight.reporting.eda import (
    build_eda_snapshot,
    eda_frames,
    run_eda_queries,
    write_eda_figures,
)
from gridsight.reporting.kpi_contract import (
    build_kpi_snapshot,
    run_kpi_queries,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

In [ ]:
engine = create_database_engine()
try:
    kpi_results = run_kpi_queries(engine)
    analysis_results = run_eda_queries(engine)
finally:
    engine.dispose()

kpi_snapshot = build_kpi_snapshot(kpi_results)
eda_snapshot = build_eda_snapshot(analysis_results)
frames = eda_frames(analysis_results)
print('Verified KPI and EDA query contracts loaded.')

## Annual and monthly movement

Energy is summed, power and price are averaged, and renewable share uses reported generation as its denominator.

In [ ]:
pd.DataFrame(kpi_snapshot['annual_kpis'])

In [ ]:
frames['monthly_series'][
    [
        'month_start',
        'average_grid_load_gw',
        'renewable_share_of_reported_generation_percent',
        'average_day_ahead_price_eur_per_mwh',
        'negative_price_hour_count',
    ]
]

## Local-hour load shape

The 48-row contract separates weekdays and weekends for each Europe/Berlin clock hour. Percentile bands describe dispersion; they are not forecast intervals.

In [ ]:
frames['load_shape'].pivot(
    index='hour_key',
    columns='day_type',
    values='average_grid_load_gw',
)

## Daily associations and unusual periods

Pearson coefficients summarize linear daily association. Year-specific values are shown because an all-period coefficient can mix different price regimes. Unusual days are selected by six declared ranking rules, not by an anomaly model.

In [ ]:
pd.DataFrame(eda_snapshot['daily_correlations'])

In [ ]:
pd.DataFrame(eda_snapshot['unusual_days'])

## Reproducible figures

In [ ]:
figure_paths = write_eda_figures(
    kpi_snapshot,
    analysis_results,
    PROJECT_ROOT / 'reports' / 'figures',
)
for figure_path in figure_paths:
    display(Image(filename=str(figure_path)))

## Interpretation boundary

The figures support descriptive portfolio findings only. Grid load and reported generation are different system concepts; unavailable generation is not zero; DE/LU wholesale prices are not retail prices; and no relationship shown here establishes causation.